# Online Retail — EDA, Cleaning, Apriori & Cross-Sell Evaluation

This notebook walks the whole project end to end: explore the raw data, apply
the cleaning policy, build baskets, mine association rules with Apriori,
evaluate the result against a popularity baseline, and run the revenue
simulation.

**It imports the `crosssell` package rather than re-implementing anything.**
Every transformation you see here is the same code the API serves and the test
suite covers, so the notebook and the production pipeline cannot drift apart.

Runs in **Google Colab** or locally. Colab users: run the setup cell, which
clones the project and downloads the dataset for you.

## 1. Setup

In [ ]:
import sys, subprocess, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install dependencies and fetch the project + dataset.
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "pandas", "numpy", "pyyaml", "matplotlib", "openpyxl", "mlxtend"], check=True)
    if not Path("/content/crosssell-project").exists():
        # Replace with your own repository URL if you have pushed this project.
        print("Upload the project folder to /content/crosssell-project, or set PROJECT_ROOT below.")
    PROJECT_ROOT = Path("/content/crosssell-project")
else:
    # Local: the notebook lives in <project>/notebooks/
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", PROJECT_ROOT)
print("Package importable:", (SRC / "crosssell" / "__init__.py").exists())

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import crosssell
from crosssell.config import load_config

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# One house style for every chart in the notebook: single hue, recessive grid.
BLUE, ORANGE, GRID, INK = "#2a78d6", "#eb6834", "#e8e8e4", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRID, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK, "figure.dpi": 110,
})

print("crosssell", crosssell.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

### Load the configuration

`config/config.yaml` holds every tunable in the project — the cleaning policy,
the split, the mining thresholds, the ranking order. It is copied verbatim into
each rule set's manifest, which is what makes a result traceable.

In [ ]:
cfg = load_config(PROJECT_ROOT / "config" / "config.yaml")
print("mining        :", cfg.mining)
print()
print("recommendation:", cfg.recommendation)
print()
print("config fingerprint:", cfg.fingerprint())

## 2. Load the raw dataset

The raw file is treated as immutable: nothing below writes to it. If the file is
missing, the next cell downloads it from the UCI Machine Learning Repository
(the original source of the Kaggle dataset).

In [ ]:
from crosssell.data.loader import file_fingerprint, load_raw

raw_path = cfg.raw_data_path
if not raw_path.exists():
    print("Dataset not found, downloading…")
    subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "download_data.py")], check=True)

fingerprint = file_fingerprint(raw_path)
df_raw = load_raw(raw_path)

print(f"File   : {fingerprint['name']} ({fingerprint['bytes']/1e6:.1f} MB)")
print(f"sha256 : {fingerprint['sha256']}")
print(f"Shape  : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

### What the columns mean

| Column | Role | Treatment |
| --- | --- | --- |
| `InvoiceNo` | transaction id | basket key; a `C` prefix marks a cancellation |
| `StockCode` | product id | the canonical item identity for mining |
| `Description` | product name | display only |
| `Quantity` | units | binary presence for mining; negatives are returns |
| `InvoiceDate` | timestamp | chronological split and trend analysis |
| `UnitPrice` | price | revenue analysis and the simulator |
| `CustomerID` | customer | optional; cohorts. Missing does **not** invalidate a basket |
| `Country` | geography | optional segmentation |

One row is a product line, not a complete order. Turning lines into baskets is
the feature-engineering step further down.

In [ ]:
overview = pd.DataFrame({
    "metric": ["rows", "columns", "unique invoices (incl. cancellations)", "unique stock codes",
               "unique customers", "countries", "exact duplicate rows"],
    "value": [len(df_raw), df_raw.shape[1], df_raw["InvoiceNo"].nunique(),
              df_raw["StockCode"].nunique(), df_raw["CustomerID"].nunique(),
              df_raw["Country"].nunique(), int(df_raw.duplicated().sum())],
})
display(overview)
df_raw.info()

## 3. Data quality

In [ ]:
missing = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
}).sort_values("missing_percent", ascending=False)
display(missing)

ax = missing["missing_percent"].plot(kind="barh", color=BLUE, figsize=(7, 3.4))
ax.set_title("Missing values by column")
ax.set_xlabel("% of rows missing")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

A quarter of rows have no `CustomerID`. That is fine for this project: Apriori
works at the **invoice** level, and an anonymous invoice is still a valid basket.
It only limits the cohort analysis at the end.

In [ ]:
from crosssell.data.preprocessing import is_cancellation, non_product_mask

work = df_raw.copy()
work["InvoiceNo"] = work["InvoiceNo"].astype("string")
work["StockCode"] = work["StockCode"].astype("string")

exceptions = pd.DataFrame({
    "measure": ["invoice starts with C", "negative quantity", "zero quantity",
                "negative unit price", "zero unit price", "non-product service code"],
    "rows": [int(is_cancellation(work["InvoiceNo"]).sum()),
             int((work["Quantity"] < 0).sum()), int((work["Quantity"] == 0).sum()),
             int((work["UnitPrice"] < 0).sum()), int((work["UnitPrice"] == 0).sum()),
             int(non_product_mask(work["StockCode"], cfg.preprocessing).sum())],
})
display(exceptions)

ax = exceptions.set_index("measure")["rows"].plot(kind="barh", color=BLUE, figsize=(7, 3))
ax.set_title("Transaction exceptions in the raw file")
ax.set_xlabel("rows"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

Those "non-product service codes" are the interesting ones: `POST`, `DOT`, `M`,
`BANK CHARGES`, `gift_*`. Postage co-occurs with nearly everything, so leaving
it in the data produces rules like *"buy X → buy POST"* — statistically strong,
commercially useless.

In [ ]:
display(df_raw[["Quantity", "UnitPrice"]].describe(percentiles=[.01, .25, .5, .75, .99]))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
q = df_raw["Quantity"]
axes[0].hist(q[(q > q.quantile(0.01)) & (q < q.quantile(0.99))], bins=60, color=BLUE)
axes[0].set_title("Quantity (1st–99th percentile)"); axes[0].set_xlabel("units")
p = df_raw["UnitPrice"]
axes[1].hist(p[(p > p.quantile(0.01)) & (p < p.quantile(0.99))], bins=60, color=BLUE)
axes[1].set_title("Unit price (1st–99th percentile)"); axes[1].set_xlabel("price")
plt.tight_layout(); plt.show()

Both distributions are heavily skewed with legitimate extremes — a bulk order of
1,000 units is a real wholesale order, not an error. **Do not delete an outlier
just because it is large.** The cleaning policy below removes rows that are not
purchases, not rows that are merely unusual.

## 4. Apply the cleaning policy

This is `crosssell.data.preprocessing.clean_transactions` — the same function
the pipeline runs. Every stage reports how many rows it removed.

In [ ]:
from crosssell.data.preprocessing import clean_transactions, validate_clean

cleaning = clean_transactions(df_raw, cfg.preprocessing)
clean = cleaning.frame
display(cleaning.audit)

print(f"Retained {len(clean):,} of {len(df_raw):,} rows ({cleaning.retention:.1%})")
print(f"{clean['InvoiceNo'].nunique():,} invoices, {clean['StockCode'].nunique():,} products")

In [ ]:
checks = validate_clean(clean, cfg.preprocessing)
display(pd.DataFrame({"check": list(checks), "passed": list(checks.values())}))
assert all(checks.values()), "cleaning post-conditions failed"
print("All post-conditions hold.")

## 5. Exploring the cleaned data

Note the distinction between popularity **by invoice count** and **by units
sold**. Apriori cares about the first: a product present in many baskets, not a
product sold in large quantities.

In [ ]:
by_invoices = clean.groupby("StockCode")["InvoiceNo"].nunique().sort_values(ascending=False)
by_quantity = clean.groupby("StockCode")["Quantity"].sum().sort_values(ascending=False)
names = clean.groupby("StockCode")["Description"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else "")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_invoices.head(12)[::-1].rename(index=names).plot(kind="barh", ax=axes[0], color=BLUE)
axes[0].set_title("Top products by invoice count"); axes[0].set_xlabel("invoices")
by_quantity.head(12)[::-1].rename(index=names).plot(kind="barh", ax=axes[1], color=BLUE)
axes[1].set_title("Top products by units sold"); axes[1].set_xlabel("units")
plt.tight_layout(); plt.show()

In [ ]:
basket_sizes = clean.groupby("InvoiceNo")["StockCode"].nunique()
display(basket_sizes.describe(percentiles=[.5, .75, .95, .99]).to_frame("distinct products per invoice"))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(basket_sizes[basket_sizes <= basket_sizes.quantile(0.99)], bins=50, color=BLUE)
ax.set_title("Distinct products per invoice")
ax.set_xlabel("products"); ax.set_ylabel("invoices")
plt.tight_layout(); plt.show()

print(f"Single-product invoices : {(basket_sizes == 1).sum():,}  (no co-occurrence signal)")
print(f"Multi-product invoices  : {(basket_sizes >= 2).sum():,}")

In [ ]:
time_df = clean.copy()
time_df["day"] = time_df["InvoiceDate"].dt.date
time_df["month"] = time_df["InvoiceDate"].dt.to_period("M").astype(str)
time_df["weekday"] = time_df["InvoiceDate"].dt.day_name()

fig, axes = plt.subplots(3, 1, figsize=(11, 8))
time_df.groupby("day")["InvoiceNo"].nunique().plot(ax=axes[0], color=BLUE, linewidth=1.3)
axes[0].set_title("Unique invoices per day"); axes[0].set_xlabel("")
time_df.groupby("month")["InvoiceNo"].nunique().plot(kind="bar", ax=axes[1], color=BLUE)
axes[1].set_title("Unique invoices per month"); axes[1].set_xlabel("")
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
time_df.groupby("weekday")["InvoiceNo"].nunique().reindex(order).plot(kind="bar", ax=axes[2], color=BLUE)
axes[2].set_title("Unique invoices by weekday"); axes[2].set_xlabel("")
plt.tight_layout(); plt.show()

print("Date range:", clean["InvoiceDate"].min(), "→", clean["InvoiceDate"].max())

Volume climbs steeply into November — this is a gift wholesaler feeding the
Christmas season, and no Saturday trading. Two consequences for the model:
seasonality is baked into the rules, and a **chronological** split is the honest
way to evaluate, because mining on the Christmas peak and testing on January
would flatter the model.

In [ ]:
country = clean.groupby("Country")["InvoiceNo"].nunique().sort_values(ascending=False)
display(country.head(10).to_frame("invoices"))
print(f"United Kingdom share of invoices: {country.get('United Kingdom', 0)/country.sum():.1%}")

## 6. Feature engineering: baskets, catalog, splits

A basket is the **set of distinct products** in one invoice. Buying ten of
something is still one co-occurrence.

In [ ]:
from crosssell.data.baskets import build_product_catalog, build_transactions, split_transactions

catalog = build_product_catalog(clean)
transactions = build_transactions(clean, cfg.preprocessing, cfg.cohorts)

display(catalog.head(8))
display(pd.Series(transactions.describe()).to_frame("baskets"))

In [ ]:
splits = split_transactions(transactions, cfg.split, seed=cfg.evaluation.random_seed)

# Product eligibility is decided on the TRAINING window only -- deciding it on
# the full dataset would leak which products survive into the future.
train_counts = splits["train"].item_counts()
eligible = train_counts[train_counts >= cfg.preprocessing.min_product_invoices].index
splits = {name: part.restrict_products(eligible, cfg.preprocessing.min_basket_size)
          for name, part in splits.items()}

display(pd.DataFrame({name: part.describe() for name, part in splits.items()}).T)
print(f"Products eligible for mining: {len(eligible):,} of {len(train_counts):,}")

## 7. Apriori

`crosssell.mining.apriori` implements the algorithm from Agrawal & Srikant,
*Fast Algorithms for Mining Association Rules in Large Databases* (VLDB 1994):
candidates are generated by joining `L(k-1)` with itself and pruned with the
downward-closure property (every subset of a frequent itemset is frequent).

The test-suite asserts this implementation returns exactly the same itemsets and
counts as `mlxtend`.

In [ ]:
import time
from crosssell.mining.apriori import apriori, min_support_count, support_counts_to_frame

train = splits["train"]
threshold = min_support_count(len(train), cfg.mining.min_support)
print(f"min_support {cfg.mining.min_support} of {len(train):,} baskets = at least {threshold} invoices")

t0 = time.perf_counter()
frequent = apriori(train.itemsets, min_support=cfg.mining.min_support,
                   max_len=cfg.mining.max_itemset_size, verbose=True)
print(f"\nMined {len(frequent):,} frequent itemsets in {time.perf_counter()-t0:.1f}s")

itemsets = support_counts_to_frame(frequent, len(train))
display(itemsets["length"].value_counts().sort_index().to_frame("itemsets"))
display(itemsets.head(8))

### Cross-check against mlxtend

A hand-written algorithm is only trustworthy if an independent implementation
agrees with it.

In [ ]:
try:
    from mlxtend.frequent_patterns import apriori as mlx_apriori
    matrix = train.to_bool_frame()
    reference = mlx_apriori(matrix, min_support=cfg.mining.min_support,
                            use_colnames=True, max_len=cfg.mining.max_itemset_size)
    expected = {frozenset(r.itemsets): round(r.support * len(train))
                for r in reference.itertuples(index=False)}
    print("same itemsets :", set(frequent) == set(expected))
    print("same counts   :", frequent == expected)
except ImportError:
    print("mlxtend not installed - skipping the cross-check")

## 8. Association rules

For a rule `A → B`:

| metric | definition | reading |
| --- | --- | --- |
| support | `P(A ∪ B)` | how common the combination is |
| confidence | `P(A ∪ B) / P(A)` | P(B given A) |
| lift | `confidence / P(B)` | strength relative to B's own popularity |
| leverage | `P(A ∪ B) − P(A)P(B)` | absolute excess over independence |

`lift > 1` is the bar for calling something a cross-sell signal.

In [ ]:
from crosssell.mining.rules import generate_rules

rules = generate_rules(
    frequent, n_transactions=len(train),
    min_confidence=cfg.mining.min_confidence,
    min_lift=cfg.mining.min_lift,
    min_antecedent_count=cfg.mining.min_antecedent_count,
    max_consequent_size=cfg.mining.max_consequent_size,
)
print(f"{len(rules):,} rules passed the thresholds")

name_of = catalog.set_index("StockCode")["ProductName"].to_dict()
readable = rules.head(10).copy()
readable["if"] = readable["antecedents"].map(lambda a: " + ".join(name_of.get(i, i) for i in a))
readable["then"] = readable["consequents"].map(lambda c: " + ".join(name_of.get(i, i) for i in c))
display(readable[["if", "then", "support", "confidence", "lift", "support_count"]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, metric, clip in zip(axes, ["support", "confidence", "lift"], [0.99, 1.0, 0.99]):
    values = rules[metric]
    ax.hist(values.clip(upper=values.quantile(clip)), bins=30, color=BLUE)
    ax.set_title(metric.capitalize()); ax.set_xlabel(metric); ax.set_ylabel("rules")
plt.tight_layout(); plt.show()

display(rules[["support", "confidence", "lift", "leverage"]].describe(percentiles=[.25, .5, .75, .95]))

Notice the shape: most rules sit at modest confidence with lift between 2 and 5,
and a thin tail of very high lift. Those extreme-lift rules are usually **colour
variants of the same product** — statistically the strongest association in the
data, commercially a substitute rather than an add-on. This is why the engine
uses lift as an admission filter and ranks by confidence.

## 9. Serving recommendations

The `Recommender` indexes rules by antecedent. A rule fires only when its whole
"if" side is in the basket, products already in the basket are never recommended
back, and when nothing clears the thresholds it returns a clearly labelled
popularity fallback rather than a fabricated confidence.

In [ ]:
from crosssell.service.recommender import Recommender

recommender = Recommender(
    rules=rules,
    catalog=catalog[catalog["StockCode"].isin(set(eligible))].reset_index(drop=True),
    popularity=train.item_support(),
    cfg=cfg.recommendation,
    ruleset_version="notebook",
)

def show(basket, top_k=5):
    print("Basket:", ", ".join(f"{p} ({recommender.product_name(p)})" for p in basket))
    for rec in recommender.recommend(basket, top_k=top_k):
        metrics = (f"conf={rec.confidence:.3f} lift={rec.lift:.2f}"
                   if rec.source == "association_rule" else f"popularity={rec.popularity_support:.4f}")
        print(f"  {rec.rank}. {rec.product_name[:44]:<46} {metrics:<28} [{rec.source}]")
    print()

show(["85123A"])                 # the best-selling product
show(["85123A", "71053"])        # a two-product basket

In [ ]:
# A product with no qualifying rule falls back -- and says so.
rare = (train.item_counts()[lambda s: s < 20]).index[0]
show([rare], top_k=3)

## 10. Evaluation

The honest question is not "do the rules look plausible" but "does this beat the
obvious alternative". The obvious alternative is recommending the best-sellers.

For each hold-out basket we hide one product (fixed seed), recommend from the
rest, and check whether the hidden product comes back in the top K. The same
replay runs for the popularity baseline.

In [ ]:
from crosssell.evaluation import leave_one_out_replay

rows = []
for k in (1, 3, 5, 10, 20):
    res = leave_one_out_replay(recommender, splits["test"], k=k, seed=cfg.evaluation.random_seed)
    rows.append({"k": k, "hit_rate": res["hit_rate_at_k"],
                 "popularity_baseline": res["popularity_baseline_hit_rate"],
                 "uplift": res["uplift_vs_popularity"], "mrr": res["mrr_at_k"],
                 "basket_coverage": res["basket_coverage"]})
curve = pd.DataFrame(rows)
display(curve)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(curve["k"], curve["hit_rate"], marker="o", color=BLUE, linewidth=2, label="Association rules")
ax.plot(curve["k"], curve["popularity_baseline"], marker="s", linestyle="--", color=ORANGE,
        linewidth=2, label="Popularity baseline")
ax.set_xlabel("K"); ax.set_ylabel("hit rate"); ax.set_xticks(curve["k"])
ax.set_title("Hold-out hit rate by list length")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

**This is the headline number of the project.** The rules recover the hidden
product roughly eight times more often than recommending best-sellers, on
invoices from a period the miner never saw.

In [ ]:
coverage = recommender.coverage()
print(f"Catalog coverage: {coverage['coverage']:.1%} "
      f"({coverage['products_with_recommendation']:,} of {coverage['products_tested']:,} products "
      "have at least one rule)")
print("The remainder fall back to best-sellers, explicitly labelled as a fallback.")

### Choosing thresholds without cheating

Thresholds are tuned on the **validation** split. The test split above is
evaluated once, at the end. Tuning on the data you report is how offline
numbers end up looking better than reality.

In [ ]:
from crosssell.evaluation import threshold_sweep

sweep = threshold_sweep(recommender, splits["validation"],
                        confidences=[0.15, 0.25, 0.35], lifts=[2.0, 3.0],
                        k=cfg.evaluation.k, seed=cfg.evaluation.random_seed, max_baskets=2000)
display(sweep)

n = int(sweep["baskets_evaluated"].iloc[0]); best = float(sweep["hit_rate_at_k"].iloc[0])
print(f"One standard error ≈ {np.sqrt(best*(1-best)/n):.4f} — rows within that band are a tie.")

## 11. Revenue-impact simulation

```
expected additional orders    = eligible transactions × adoption rate
estimated incremental revenue = expected additional orders × average order value
```

Every number below is a **planning estimate under stated assumptions**, not
measured uplift.

In [ ]:
from crosssell.service.simulation import simulate_revenue

replay = leave_one_out_replay(recommender, splits["test"], k=cfg.evaluation.k,
                              seed=cfg.evaluation.random_seed)
eligible_transactions = int(round(len(splits["test"]) * replay["basket_coverage"]))

# Average value of ONE purchase of a recommended product, weighted by how often
# each product is actually recommended. This is what `cli train` stores as the
# simulator's default; a plain median over the catalog would under-count,
# because the products the rules surface are not the median product.
weights = rules["support_count"].astype(float)
values = rules["consequents"].map(lambda c: c[0]).map(
    catalog.set_index("StockCode")["avg_line_revenue"]).astype(float)
valid = values.notna()
aov = float((values[valid] * weights[valid]).sum() / weights[valid].sum())

report = simulate_revenue(eligible_transactions, round(aov, 2), cfg.simulation.scenarios)
scenarios = pd.DataFrame([s.to_dict() for s in report.scenarios])
display(scenarios[["name", "adoption_rate", "eligible_transactions",
                   "expected_additional_orders", "estimated_incremental_revenue"]])

ax = scenarios.plot(kind="bar", x="name", y="estimated_incremental_revenue",
                    color=BLUE, legend=False, figsize=(6, 3))
ax.set_title("Simulated incremental revenue by scenario")
ax.set_ylabel("£"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

print(report.disclaimer)

## 12. Stretch goal: new vs repeat customers

A customer is "new" on their first invoice and "repeat" from the second onwards.
The label uses the invoice sequence **at the time of that invoice**, so it never
uses information from the future.

In [ ]:
cohort_rules = {}
for cohort in ("new", "repeat"):
    subset = train.by_cohort(cohort)
    if len(subset) < 100:
        print(f"{cohort}: only {len(subset)} baskets, skipping")
        continue
    freq = apriori(subset.itemsets, min_support=cfg.mining.min_support,
                   max_len=cfg.mining.max_itemset_size)
    cohort_rules[cohort] = generate_rules(
        freq, len(subset), min_confidence=cfg.mining.min_confidence,
        min_lift=cfg.mining.min_lift, min_antecedent_count=cfg.mining.min_antecedent_count,
        max_consequent_size=cfg.mining.max_consequent_size)
    print(f"{cohort:<7} {len(subset):>6,} baskets -> {len(cohort_rules[cohort]):>6,} rules")

if cohort_rules:
    display(pd.DataFrame({
        c: {"rules": len(r), "avg confidence": r["confidence"].mean(), "avg lift": r["lift"].mean()}
        for c, r in cohort_rules.items()
    }).T)

## 13. From notebook to production

Everything above is one command:

```bash
python -m crosssell.cli train     # clean → baskets → Apriori → rules → evaluate → version
python -m crosssell.cli report    # the written evaluation report with figures
python -m crosssell.cli serve     # the FastAPI service the dashboard talks to
```

`train` additionally writes a versioned rule set with a reproducibility
manifest — dataset hash, full configuration, environment, mining statistics and
evaluation summary — so any recommendation the API serves can be traced back to
the exact data and settings that produced it.

### Limitations to carry forward

* Association is not causation: a rule says two products were bought together.
* Simulated revenue multiplies assumptions; it is not evidence of uplift.
* One UK gift wholesaler, Dec 2010 – Dec 2011. Seasonality is in the rules.
* Hit rate measures recovery of a product the customer *did* buy; it cannot show
  whether a recommendation would have *created* a purchase.